In [1]:
%load_ext autoreload
%autoreload 2
import sys, json, time
from pathlib import Path
import numpy as np
import polars as pl

ROOT = Path.cwd()
while ROOT != ROOT.parent and not ((ROOT / 'src').is_dir() and (ROOT / 'configs').is_dir()):
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from src.config import load_config
from src.features.prepare import META
cfg = load_config()
PROC, REPORTS = ROOT / 'data' / 'processed', ROOT / 'reports'
SEED = cfg['seed']

lf = pl.scan_parquet(PROC / 'model_table.parquet')
schema = lf.collect_schema()
FEATS_ALL = [c for c in schema.names() if c not in META]

# R3 mở rộng (28/10): proxy thời gian pmts_year_1139T + 12 *_yrs_ago
R3_EXT = ([c for c in FEATS_ALL if c.startswith('credit_bureau_a_2__pmts_year_1139T__')]
          + [c for c in FEATS_ALL if c.endswith('_yrs_ago')])
FEATS = [c for c in FEATS_ALL if c not in R3_EXT]
print('R3 mở rộng loại:', len(R3_EXT), '| còn:', len(FEATS))

# R6: Champion loại PSI(Valid so với cửa sổ Train) > 0.25
psi = pl.read_csv(REPORTS / 'psi_valid_vs_window.csv').filter(pl.col('feature').is_in(FEATS))
R6 = {w: psi.filter(pl.col(w) > 0.25)['feature'].to_list() for w in ['A', 'B']}
CHAMP_FEATS = {w: [c for c in FEATS if c not in R6[w]] for w in ['A', 'B']}
for w in ['A', 'B']:
    print(w, '| R6 loại:', len(R6[w]), '| vào optbinning:', len(CHAMP_FEATS[w]))

(REPORTS / 'champion_feature_lists.json').write_text(
    json.dumps({'R3_ext': R3_EXT, 'R6': R6, 'champion_feats': CHAMP_FEATS}, indent=1),
    encoding='utf-8')

import optbinning
print('optbinning', optbinning.__version__)

R3 mở rộng loại: 15 | còn: 628
A | R6 loại: 73 | vào optbinning: 555
B | R6 loại: 16 | vào optbinning: 612
optbinning 1.0.0


In [2]:
import os
from joblib import Parallel, delayed
from optbinning import OptimalBinning

WIN = {'A': (0, 43), 'B': (20, 43)}
N_JOBS = max(1, (os.cpu_count() or 2) - 1)
MODEL_DIR = ROOT / 'models'
MODEL_DIR.mkdir(exist_ok=True)

def load_train(w):
    w0, w1 = WIN[w]
    return (lf.filter((pl.col('split') == 'train') & pl.col('WEEK_NUM').is_between(w0, w1))
              .select(['target'] + CHAMP_FEATS[w])
              .collect(engine='streaming'))

def col_arrays(d, cols):
    for c in cols:
        s = d.get_column(c)
        is_cat = s.dtype == pl.String
        yield c, (s.to_numpy() if is_cat else s.cast(pl.Float64).to_numpy()), is_cat

def fit_one(name, x, y, is_cat):
    t0 = time.time()
    ob = OptimalBinning(
        name=name, dtype='categorical' if is_cat else 'numerical', solver='cp',
        max_n_prebins=20, min_prebin_size=0.05, min_bin_size=0.05,
        monotonic_trend=None if is_cat else 'auto_asc_desc',
        cat_cutoff=0.01 if is_cat else None, time_limit=60,
    )
    ob.fit(x, y)
    bt = ob.binning_table.build()
    return name, ob, dict(
        feature=name, dtype='cat' if is_cat else 'num', status=ob.status,
        n_bins=len(bt) - 3,                      # trừ các dòng Special, Missing, Totals
        iv=float(bt.loc['Totals', 'IV']), seconds=round(time.time() - t0, 2),
    )

def fit_many(d, cols):
    y = d['target'].to_numpy()
    out = Parallel(n_jobs=N_JOBS)(delayed(fit_one)(c, x, y, cat) for c, x, cat in col_arrays(d, cols))
    return {n: ob for n, ob, _ in out}, pl.DataFrame([r for _, _, r in out])

trB = load_train('B')
print(trB.shape, f'{trB.estimated_size("gb"):.2f} GB', '| N_JOBS =', N_JOBS)

t0 = time.time()
_, test = fit_many(trB, CHAMP_FEATS['B'][:24])
sec = time.time() - t0
print(f'24 feature: {sec:.1f}s → ước tính B: {sec * 612 / 24 / 60:.1f} phút, '
      f'A: {sec * 555 / 24 / 60 * 841500 / 522720:.1f} phút')
print(test.sort('iv', descending=True))

(522720, 613) 1.24 GB | N_JOBS = 11
24 feature: 14.3s → ước tính B: 6.1 phút, A: 8.9 phút
shape: (24, 6)
┌────────────────────────────────┬───────┬─────────┬────────┬──────────┬─────────┐
│ feature                        ┆ dtype ┆ status  ┆ n_bins ┆ iv       ┆ seconds │
│ ---                            ┆ ---   ┆ ---     ┆ ---    ┆ ---      ┆ ---     │
│ str                            ┆ str   ┆ str     ┆ i64    ┆ f64      ┆ f64     │
╞════════════════════════════════╪═══════╪═════════╪════════╪══════════╪═════════╡
│ avgdpdtolclosure24_3658938P    ┆ num   ┆ OPTIMAL ┆ 4      ┆ 0.330594 ┆ 0.48    │
│ avgdbddpdlast24m_3658932P      ┆ num   ┆ OPTIMAL ┆ 7      ┆ 0.257562 ┆ 1.01    │
│ avgmaxdpdlast9m_3716943P       ┆ num   ┆ OPTIMAL ┆ 2      ┆ 0.156351 ┆ 0.28    │
│ avgdbddpdlast3m_4187120P       ┆ num   ┆ OPTIMAL ┆ 5      ┆ 0.149264 ┆ 0.45    │
│ amtinstpaidbefduel24m_4187115A ┆ num   ┆ OPTIMAL ┆ 7      ┆ 0.099808 ┆ 1.83    │
│ …                              ┆ …     ┆ …       ┆ …      ┆ …  

In [3]:
import joblib

IV_MIN = cfg['champion']['iv_min']
print('IV_MIN =', IV_MIN)

BINS, IV = {}, {}
for w in ['B', 'A']:
    t0 = time.time()
    d = trB if w == 'B' else load_train('A')
    if w == 'B':
        del trB
    BINS[w], IV[w] = fit_many(d, CHAMP_FEATS[w])
    joblib.dump(BINS[w], MODEL_DIR / f'champion_binning_{w}.joblib')
    IV[w].write_csv(REPORTS / f'champion_iv_{w}.csv')
    print(f'{w}: {(time.time() - t0) / 60:.1f} phút | n = {d.height}'
          f' | không OPTIMAL: {IV[w].filter(pl.col("status") != "OPTIMAL").height}'
          f' | IV > {IV_MIN}: {IV[w].filter(pl.col("iv") > IV_MIN).height}'
          f' | IV > 0.5: {IV[w].filter(pl.col("iv") > 0.5).height}')
    del d

iv_cmp = (
    IV['A'].select('feature', pl.col('iv').alias('iv_A'))
    .join(IV['B'].select('feature', pl.col('iv').alias('iv_B')), on='feature', how='full', coalesce=True)
    .sort('iv_B', descending=True, nulls_last=True)
)
iv_cmp.write_csv(REPORTS / 'champion_iv_compare.csv')
print(iv_cmp.head(25))
print(IV['B'].group_by('dtype').agg(pl.len().alias('n'), (pl.col('iv') > IV_MIN).sum().alias('n_pass')))

IV_MIN = 0.02
B: 0.7 phút | n = 522720 | không OPTIMAL: 12 | IV > 0.02: 252 | IV > 0.5: 0
A: 1.0 phút | n = 841500 | không OPTIMAL: 12 | IV > 0.02: 192 | IV > 0.5: 0
shape: (25, 3)
┌─────────────────────────────────┬──────────┬──────────┐
│ feature                         ┆ iv_A     ┆ iv_B     │
│ ---                             ┆ ---      ┆ ---      │
│ str                             ┆ f64      ┆ f64      │
╞═════════════════════════════════╪══════════╪══════════╡
│ credit_bureau_a_2__pmts_dpd_30… ┆ null     ┆ 0.390113 │
│ credit_bureau_a_1__dpdmax_757P… ┆ null     ┆ 0.382076 │
│ credit_bureau_a_1__numberofove… ┆ null     ┆ 0.375148 │
│ credit_bureau_a_2__pmts_overdu… ┆ null     ┆ 0.361406 │
│ credit_bureau_a_1__overdueamou… ┆ null     ┆ 0.339538 │
│ …                               ┆ …        ┆ …        │
│ credit_bureau_a_2__pmts_overdu… ┆ null     ┆ 0.259159 │
│ avgdbddpdlast24m_3658932P       ┆ 0.227394 ┆ 0.257562 │
│ credit_bureau_a_2__pmts_overdu… ┆ 0.245063 ┆ 0.254811 │
│ credi

In [ ]:
bad = (
    pl.concat([IV[w].with_columns(pl.lit(w).alias('window')) for w in ['A', 'B']])
    .filter(pl.col('status') != 'OPTIMAL')
    .sort(['feature', 'window'])
)
print(bad.select('window', 'feature', 'dtype', 'status', 'n_bins', 'iv', 'seconds'))
print(bad.group_by('window', 'status').len())

In [4]:
CORR_MAX = 0.7

def load_rows(feats, cond):
    return lf.filter(cond).select(['WEEK_NUM', 'target'] + feats).collect(engine='streaming')

def woe_matrix(bins, feats, d):
    cols = []
    for c in feats:
        s = d.get_column(c)
        x = s.to_numpy() if s.dtype == pl.String else s.cast(pl.Float64).to_numpy()
        cols.append(bins[c].transform(x, metric='woe', metric_missing='empirical',
                                      metric_special='empirical').astype(np.float32))
    return np.column_stack(cols)

DATA = {}
for w in ['B', 'A']:
    t0 = time.time()
    sel = IV[w].filter(pl.col('iv') > IV_MIN).sort('iv', descending=True)['feature'].to_list()
    w0, w1 = WIN[w]
    tr = load_rows(sel, (pl.col('split') == 'train') & pl.col('WEEK_NUM').is_between(w0, w1))
    va = load_rows(sel, pl.col('split') == 'valid')
    Xtr, Xva = woe_matrix(BINS[w], sel, tr), woe_matrix(BINS[w], sel, va)

    C = np.abs(np.corrcoef(Xtr, rowvar=False))
    keep = []
    for i in range(len(sel)):
        if all(C[i, j] < CORR_MAX for j in keep):
            keep.append(i)

    DATA[w] = dict(
        feats=[sel[i] for i in keep],
        Xtr=Xtr[:, keep], ytr=tr['target'].to_numpy(),
        Xva=Xva[:, keep], yva=va['target'].to_numpy(), wk_va=va['WEEK_NUM'].to_numpy(),
    )
    print(f'{w}: IV > {IV_MIN}: {len(sel)} → sau lọc tương quan: {len(keep)}'
          f' | Train {Xtr.shape} | Valid {Xva.shape} | {(time.time() - t0) / 60:.1f} phút')
    del tr, va, Xtr, Xva, C

(REPORTS / 'champion_selected_after_corr.json').write_text(
    json.dumps({w: DATA[w]['feats'] for w in DATA}, indent=1), encoding='utf-8')

B: IV > 0.02: 252 → sau lọc tương quan: 79 | Train (522720, 252) | Valid (224421, 252) | 0.4 phút
A: IV > 0.02: 192 → sau lọc tương quan: 72 | Train (841500, 192) | Valid (224421, 192) | 0.5 phút


5507

In [5]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

def gini(y, p):
    return 2 * roc_auc_score(y, p) - 1

def psi(expected, actual, n_bins=10):
    qs = np.unique(np.quantile(expected, np.linspace(0, 1, n_bins + 1)))
    qs[0], qs[-1] = -np.inf, np.inf
    e = np.clip(np.histogram(expected, qs)[0] / len(expected), 1e-6, None)
    a = np.clip(np.histogram(actual, qs)[0] / len(actual), 1e-6, None)
    return float(np.sum((a - e) * np.log(a / e)))

def fit_logit(X, y):
    return LogisticRegression(C=1.0, solver='lbfgs', max_iter=2000).fit(X, y)

CHAMP, rows = {}, []
for w in ['B', 'A']:
    D, t0 = DATA[w], time.time()
    feats = D['feats']                                      # đã sắp theo IV giảm dần
    exp_sign = np.sign(np.corrcoef(D['Xtr'][:, 0], D['ytr'])[0, 1])
    idx, dropped = list(range(len(feats))), []
    while True:
        m = fit_logit(D['Xtr'][:, idx], D['ytr'])
        wrong = [k for k, b in enumerate(m.coef_[0]) if np.sign(b) != exp_sign]
        if not wrong:
            break
        k = max(wrong)                                      # sai dấu, IV thấp nhất
        dropped.append(feats[idx[k]])
        idx.pop(k)

    p_tr = m.predict_proba(D['Xtr'][:, idx])[:, 1]
    p_va = m.predict_proba(D['Xva'][:, idx])[:, 1]
    wk = D['wk_va']
    g_wk = [gini(D['yva'][wk == u], p_va[wk == u]) for u in np.unique(wk)]
    CHAMP[w] = dict(model=m, feats=[feats[i] for i in idx], dropped_sign=dropped)
    joblib.dump(CHAMP[w], MODEL_DIR / f'champion_logit_{w}.joblib')

    (pl.DataFrame({'feature': CHAMP[w]['feats'], 'coef': m.coef_[0]})
       .join(IV[w].select('feature', 'iv'), on='feature')
       .sort('iv', descending=True)
       .write_csv(REPORTS / f'champion_coef_{w}.csv'))

    rows.append(dict(
        window=w, exp_sign=int(exp_sign), n_feat=len(idx), n_dropped_sign=len(dropped),
        gini_train=gini(D['ytr'], p_tr), gini_valid=gini(D['yva'], p_va),
        gap=gini(D['ytr'], p_tr) - gini(D['yva'], p_va), psi_score=psi(p_tr, p_va),
        week_mean=float(np.mean(g_wk)), week_std=float(np.std(g_wk)), week_min=float(np.min(g_wk)),
        minutes=round((time.time() - t0) / 60, 1),
    ))
    print(w, 'xong', rows[-1]['minutes'], 'phút | bỏ do sai dấu:', len(dropped))

res = pl.DataFrame(rows)
res.write_csv(REPORTS / 'champion_window_ab.csv')
pl.Config.set_tbl_cols(20)
print(res)

B xong 0.9 phút | bỏ do sai dấu: 17
A xong 1.4 phút | bỏ do sai dấu: 15
shape: (2, 12)
┌────────┬────────┬────────┬───────┬───────┬───────┬───────┬───────┬───────┬───────┬───────┬───────┐
│ window ┆ exp_si ┆ n_feat ┆ n_dro ┆ gini_ ┆ gini_ ┆ gap   ┆ psi_s ┆ week_ ┆ week_ ┆ week_ ┆ minut │
│ ---    ┆ gn     ┆ ---    ┆ pped_ ┆ train ┆ valid ┆ ---   ┆ core  ┆ mean  ┆ std   ┆ min   ┆ es    │
│ str    ┆ ---    ┆ i64    ┆ sign  ┆ ---   ┆ ---   ┆ f64   ┆ ---   ┆ ---   ┆ ---   ┆ ---   ┆ ---   │
│        ┆ i64    ┆        ┆ ---   ┆ f64   ┆ f64   ┆       ┆ f64   ┆ f64   ┆ f64   ┆ f64   ┆ f64   │
│        ┆        ┆        ┆ i64   ┆       ┆       ┆       ┆       ┆       ┆       ┆       ┆       │
╞════════╪════════╪════════╪═══════╪═══════╪═══════╪═══════╪═══════╪═══════╪═══════╪═══════╪═══════╡
│ B      ┆ -1     ┆ 62     ┆ 17    ┆ 0.632 ┆ 0.617 ┆ 0.015 ┆ 0.003 ┆ 0.615 ┆ 0.026 ┆ 0.581 ┆ 0.9   │
│        ┆        ┆        ┆       ┆ 986   ┆ 459   ┆ 526   ┆ 095   ┆ 277   ┆ 714   ┆ 453   ┆       │
│ A 

In [6]:
pv = {}
for w in ['A', 'B']:
    cols = [DATA[w]['feats'].index(f) for f in CHAMP[w]['feats']]
    pv[w] = CHAMP[w]['model'].predict_proba(DATA[w]['Xva'][:, cols])[:, 1]

wk, y = DATA['B']['wk_va'], DATA['B']['yva']
assert np.array_equal(wk, DATA['A']['wk_va']) and np.array_equal(y, DATA['A']['yva'])

wkdf = pl.DataFrame([
    dict(week=int(u), gini_B=gini(y[wk == u], pv['B'][wk == u]), gini_A=gini(y[wk == u], pv['A'][wk == u]))
    for u in np.unique(wk)
]).with_columns((pl.col('gini_B') - pl.col('gini_A')).alias('diff'))
print(wkdf)
print('Số tuần B > A:', int((wkdf['diff'] > 0).sum()), '/', wkdf.height)

rng = np.random.default_rng(SEED)
diffs = []
for _ in range(200):
    i = rng.integers(0, len(y), len(y))
    diffs.append(gini(y[i], pv['B'][i]) - gini(y[i], pv['A'][i]))
lo, hi = np.percentile(diffs, [2.5, 97.5])
print(f'ΔGini Valid (B − A) = {gini(y, pv["B"]) - gini(y, pv["A"]):.4f} | CI95 bootstrap [{lo:.4f}, {hi:.4f}]')

shape: (8, 4)
┌──────┬──────────┬──────────┬──────────┐
│ week ┆ gini_B   ┆ gini_A   ┆ diff     │
│ ---  ┆ ---      ┆ ---      ┆ ---      │
│ i64  ┆ f64      ┆ f64      ┆ f64      │
╞══════╪══════════╪══════════╪══════════╡
│ 44   ┆ 0.646407 ┆ 0.637799 ┆ 0.008608 │
│ 45   ┆ 0.59838  ┆ 0.591042 ┆ 0.007338 │
│ 46   ┆ 0.592395 ┆ 0.583892 ┆ 0.008503 │
│ 47   ┆ 0.656414 ┆ 0.652381 ┆ 0.004032 │
│ 48   ┆ 0.589604 ┆ 0.588069 ┆ 0.001535 │
│ 49   ┆ 0.622841 ┆ 0.61457  ┆ 0.008271 │
│ 50   ┆ 0.581453 ┆ 0.569353 ┆ 0.0121   │
│ 51   ┆ 0.634726 ┆ 0.629128 ┆ 0.005598 │
└──────┴──────────┴──────────┴──────────┘
Số tuần B > A: 8 / 8
ΔGini Valid (B − A) = 0.0072 | CI95 bootstrap [0.0052, 0.0099]
